# 04. Permutation importance (Figure 2, Supplementary Table 7)

Each predictor, or each predefined clinical block, is shuffled in the raw test data (`N_REP` = 100 times)
while the other predictors are unchanged; the data are re-encoded and the C-index decrease and IBS increase
are recorded. Mean and 95% interval (2.5th-97.5th percentile) over the repetitions are reported.
Baseline IBS here uses a 50-point grid up to 0.999 x the maximum test follow-up, with the test set as reference.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sksurv.util import Surv

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH    = Path("../data/dummy_data.csv")     # replace with your own data
OUTPUT_DIR   = Path("../outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
PREPROC_PATH = OUTPUT_DIR / "preprocessor.joblib"  # written by 01_data_preprocessor
MODEL_PATH   = OUTPUT_DIR / "RSF_model.joblib"     # written by 02_RSF_train
RANDOM_SEED  = 42

TIME_COL, EVENT_COL, ID_COLS = "OS", "event", ["SubjectKey"]   # OS in days

def load_splits(path=DATA_PATH):
    """Return {"Train"|"Val"|"Test": (X_raw, y)} using the column split_3."""
    df = pd.read_csv(path)
    df[EVENT_COL] = df[EVENT_COL].astype(bool)
    out = {}
    for name, key in [("Train", "train"), ("Val", "val"), ("Test", "test")]:
        d = df[df["split_3"] == key]
        out[name] = (d.drop(columns=ID_COLS + [TIME_COL, EVENT_COL]),
                     Surv.from_arrays(d[EVENT_COL], d[TIME_COL]))
    return out

In [ ]:
class Pred:
    """Predictions of a fitted RSF for one evaluated set."""
    def __init__(self, model, X, y):
        self.risk  = model.predict(X)
        self.S     = model.predict_survival_function(X, return_array=True)
        self.T     = model.unique_times_
        self.event = y["event"]
        self.time  = y["time"]

    def surv_at(self, rows, grid):
        """Step-function survival S(t) at the given times for the given rows."""
        grid = np.atleast_1d(np.asarray(grid, dtype=float))
        idx  = np.searchsorted(self.T, grid, side="right") - 1
        out  = np.ones((len(rows), len(grid)))
        ok   = idx >= 0
        out[:, ok] = self.S[rows][:, idx[ok]]
        return out

In [ ]:
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from tqdm.auto import tqdm

N_REP, SEED = 100, 2025      # reduce N_REP for a quick test run

splits = load_splits()
y_train = splits["Train"][1]
preprocessor = joblib.load(PREPROC_PATH)   # from 01_data_preprocessor.ipynb
model        = joblib.load(MODEL_PATH)     # from 02_RSF_train.ipynb
X = {s: preprocessor.transform(splits[s][0]) for s in splits}
Y = {s: splits[s][1] for s in splits}
print({s: X[s].shape for s in X})

In [ ]:
FEATURE_GROUPS = {
    "Age": ["Age"], "BMI": ["BMI"], "Gender": ["Gender"],
    "Comorbidities": ["DM", "HTN", "CAD", "Hepatic Disease", "Renal Disease",
                      "Respiratory Disease", "Malignancy"],
    "Smoking": ["Smoke Quit_days", "Smoke Amount", "Smoke Duration", "Smoke Hx"],
    "PFT": ["Spirometry FVC", "Spirometry FEV1", "Diffusion DLCO"],
    "Symptoms": [f"symptom_{i}" for i in range(1, 13)],
    "Stage": ["T_stage", "N_stage", "M_stage"],
    "Adeno type": ["Adeno type"], "resection type": ["resection type"], "Opextent": ["Opextent"],
    "Neoadjuvant therapy": ["Neoadjuvant therapy"],
    "Postop Adjuvant Therapy": ["Postop Adjuvant Therapy"],
    "Biomarkers": ["VEGF", "EGFR", "Ki-67", "p53"],
}

X_test_raw, y_test = splits["Test"]
event, time = y_test["event"], y_test["time"]
T_GRID = np.linspace(time.min(), time.max() * 0.999, 50)

def scores(X_raw):
    p = Pred(model, preprocessor.transform(X_raw), y_test)
    c = concordance_index_censored(event, time, p.risk)[0]
    S = p.surv_at(np.arange(len(time)), T_GRID)
    return c, integrated_brier_score(y_test, y_test, S, T_GRID)

C_BASE, IBS_BASE = scores(X_test_raw)
print(f"Baseline C-index = {C_BASE:.4f}, IBS = {IBS_BASE:.4f}")

def permutation_importance(cols):
    rng = np.random.default_rng(SEED)
    drops, incs = [], []
    for _ in range(N_REP):
        Xp = X_test_raw.copy()
        order = rng.permutation(len(Xp))
        for c in cols:
            Xp[c] = Xp[c].values[order]
        c_p, ibs_p = scores(Xp)
        drops.append(C_BASE - c_p); incs.append(ibs_p - IBS_BASE)
    lo_c, hi_c = np.percentile(drops, [2.5, 97.5]); lo_i, hi_i = np.percentile(incs, [2.5, 97.5])
    return {"C-index reduction": np.mean(drops), "C-index 95% CI": f"[{lo_c:.4f}, {hi_c:.4f}]",
            "IBS increase": np.mean(incs), "IBS 95% CI": f"[{lo_i:.4f}, {hi_i:.4f}]"}

In [ ]:
# Grouped features (Figure 2A)
group_df = pd.DataFrame([{"Feature group": g, **permutation_importance(cols)}
                         for g, cols in tqdm(FEATURE_GROUPS.items(), desc="groups")])
group_df = group_df.sort_values("C-index reduction", ascending=False).reset_index(drop=True)
print(group_df.round(4).to_string(index=False))

In [ ]:
# Individual features (Figure 2B)
features = sorted({c for cols in FEATURE_GROUPS.values() for c in cols})
feat_df = pd.DataFrame([{"Feature": f, **permutation_importance([f])}
                        for f in tqdm(features, desc="features")])
feat_df = feat_df.sort_values("C-index reduction", ascending=False).reset_index(drop=True)
print(feat_df.round(4).to_string(index=False))